# **EDA on bank Churned customers**

This notebook is a part of the bank churn project. We will construct and validate a random forest model with scikit-Learn. The modeling objective is to predict whether a customer will churn. We will create binary values.

## Target variable: Exited column—0 or 1.

Class balance: The data is imbalanced 80/20 (not churned/churned), but we will not perform class balancing.

## Primary evaluation metric: F1 score.

## Modeling workflow and model selection: 
The champion model will be the model with the best validation F1 score. Only the champion model will be used to predict on the test data. See the annotated decision tree notebook for details and limitations of this approach.

In [ ]:
# Import neccesary libraries for our project. 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

from sklearn.ensemble import RandomForestClassifier

# This will save our models once we fit them.
import pickle
import os

In [31]:
# This will show all columns in dataframe.
pd.set_option('display.max_columns', None)

In [32]:
# I'm going to re-route the directory to where the csv file is located.
os.chdir('/Users/nhanguyen/Library/Mobile Documents/iCloud~com~omz-software~Pythonista3/Documents/')
# This line of code needs to be revised for your device. 

# Load in the csv file. 
df = pd.read_csv('Churn_Modelling.csv')
print(df.head())

   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0  
4         790

# Feature Engineering
To prepare data for modeling, we have to clean the data, as well create needed data columns. Let's start by deleting columns we don't need, because they are unneccesary or introduce bias into our model. We will drop these columns: ['RowNumber', 'CustomerId', 'Surname', 'Gender']

In [33]:
# Let's drop columns we don't need. 

churn_df = df.drop(['RowNumber', 'CustomerId', 'Surname', 'Gender'], axis=1)

print(churn_df.head())


   CreditScore Geography  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          619    France   42       2       0.00              1          1   
1          608     Spain   41       1   83807.86              1          0   
2          502    France   42       8  159660.80              3          1   
3          699    France   39       1       0.00              2          0   
4          850     Spain   43       2  125510.82              1          1   

   IsActiveMember  EstimatedSalary  Exited  
0               1        101348.88       1  
1               1        112542.58       0  
2               0        113931.57       1  
3               0         93826.63       0  
4               1         79084.10       0  


We will use the get_dummies function to turn the Geography column binary. We will use drop_first='True', which will eliminate one category. We will keep Germany, and Spain, and if the value is not true for neither categories, we'll know the customer was in France!

In [27]:
# Now we change Geography column to integers. 
# We will only keep Spain and Germany, because if neither is true, then it's France!

churn_df2 = pd.get_dummies(churn_df, columns=['Geography'], prefix='Geo', drop_first=True, dtype=int)
print(churn_df2.head())

   CreditScore  Age  Tenure    Balance  NumOfProducts  HasCrCard  \
0          619   42       2       0.00              1          1   
1          608   41       1   83807.86              1          0   
2          502   42       8  159660.80              3          1   
3          699   39       1       0.00              2          0   
4          850   43       2  125510.82              1          1   

   IsActiveMember  EstimatedSalary  Exited  Geo_Germany  Geo_Spain  
0               1        101348.88       1            0          0  
1               1        112542.58       0            0          1  
2               0        113931.57       1            0          0  
3               0         93826.63       0            0          0  
4               1         79084.10       0            0          1  


We'll split the data using train_test_split. We will use stratify=y to ensure that 80/20 class ratio of the target variable is present in both training and test dataset.

In [34]:
# Split the data and apply train_test_split. 
# Define your X and y variables.
# We will use 25% test data, and 75% train data.

y = churn_df2['Exited']

X = churn_df2.copy()
X = X.drop('Exited', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

We will begin the Random Forest model. It is important that we define the hyperparameters. 

In [45]:
# Now we apply Random Forest classifier, and use GridsearchCV on it. 

print(X_train.shape)
print(y_train.shape)
# %%time # This will show the time this model took. 
rf = RandomForestClassifier(random_state=0, n_jobs=1)

cv_params = {'max_depth': [2,3,4,5, None],
             'min_samples_leaf': [1,2,3],
             'min_samples_split': [2,3,4],
             'max_features': [2,3,4],
             'n_estimators': [75, 100, 125, 150]}

scoring = ['accuracy', 'precision', 'recall', 'f1']
# This is where the script takes a long time. 
rf_cv = GridSearchCV(rf, cv_params, scoring=scoring, cv=5, refit='f1', n_jobs=-1)

rf_cv.fit(X_train, y_train)

print(rf_cv.best_params_)

print(rf_cv.best_score_)

(7500, 10)
(7500,)
{'max_depth': None, 'max_features': 4, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 150}
0.5833023473561427


This model has an F1 score of approximately 0.5833... which is not terrible, but we want to also capture the other scores, accuracy, recall, and precision. We will make a helper function to return a table of these scores. 

In [38]:
# Let's create a table of scores from the decision tree. 
# Define a new function to do this.


def make_results(model_name, model_object):
    cv_results = pd.DataFrame(model_object.cv_results_)
    best_est_res = cv_results.iloc[cv_results['mean_test_f1'].idxmax(), :]
    f1 = best_est_res.mean_test_f1
    accuracy = best_est_res.mean_test_accuracy
    recall = best_est_res.mean_test_recall
    precision = best_est_res.mean_test_precision
    # Now create a table of scores
    table = pd.DataFrame()
    table = pd.DataFrame({'Model': [model_name],
                          'Accuracy': [accuracy],
                          'Precision': [precision],
                          'Recall': [recall],
                          'F1': [f1]})
    return table

rf_cv_results = make_results('Random Forest CV', rf_cv)
print(rf_cv_results)

results = pd.read_csv('churn_tree_results.csv', index_col=0)
print(results)

              Model  Accuracy  Precision   Recall        F1
0  Random Forest CV  0.862133   0.758639  0.47514  0.583302
                 Model        F1    Recall  Accuracy  Precision
0  Tuned Decision Tree  0.560655  0.469255    0.8504   0.701608


We will create a validation set from training dataset. We're going to tell it exactly which rows of X_train are for training, and which rows are for validation. 

To do this, we need to make a list of length len(X_train) where each element is either a 0 or -1. A 0 in index i will indicate to GridSearchCV that index i of X_train is to be held out for validation. A -1 at a given index will indicate that that index of X_train is to be used as training data.

In [42]:
# Let's make validation set.

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, 
                                            stratify=y_train, random_state=10)

split_index = [0 if x in X_val.index else -1 for x in X_train.index]

from sklearn.model_selection import PredefinedSplit

rfv = RandomForestClassifier(random_state=0, n_jobs=1)

In [43]:
cv_params = {'max_depth': [2,3,4,5, None], 
             'min_samples_leaf': [1,2,3],
             'min_samples_split': [2,3,4],
             'max_features': [2,3,4],
             'n_estimators': [75, 100, 125, 150]
             } 

scoring = ['accuracy', 'precision', 'recall', 'f1']

custom_split = PredefinedSplit(split_index)

rf_val = GridSearchCV(rfv, cv_params, scoring=scoring, cv=custom_split, refit='f1', n_jobs=-1)

rf_val.fit(X_train, y_train)

rf_val.best_params_

rf_val_results = make_results('Random Forest Validated', rf_val)
print(rf_val_results)

# I tried to make it faster but GridSearch combined with Decision Tree or 
# Random Forest is a very brutal algorithm on big dataset.
# for faster run time, best use RandomizedSearchCV.
# This took like almost 4 minutes to run. 


                     Model  Accuracy  Precision    Recall        F1
0  Random Forest Validated  0.862667   0.771739  0.464052  0.579592
